# Week 2, day 5 (morning) — Worksheet 05 SOLUTIONS: break, continue, pass   (L05)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Several questions count how many passes a loop made. Those counts are the
answer — two loops that print the same thing can do very different amounts of
work, and one of them here does none of the work you expect.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 05 — break, continue, pass. Run this once.
names = ["ana", "bo", "cai", "dee"]

rows = [
    {"id": "r1", "amount": 10.0},
    {"id": "r2", "amount": -5.0},     # negative -- skip it
    {"id": "r3", "amount": 0.0},
    {"id": "r4", "amount": 22.5},
    {"id": "r5", "amount": -1.0},     # negative -- skip it
]

# A feed that ends with a sentinel row rather than just running out.
feed = [4.0, 9.5, -2.0, 3.0, "END", 100.0, 200.0]

grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

print(len(names), "names,", len(rows), "rows,", len(feed), "feed entries")

PART A — break: leave the loop now

### Question 1

Searching with and without `break`. -> `found cai after 3 names`, then `without break, looked at 4`.

`break` leaves the loop immediately — not the `if`, the whole loop. The
fourth name is never looked at.

On four names, three passes against four is nothing. On four million rows
it is the difference between a search and a full scan, and the second loop
is worse than slow: it keeps overwriting `found`, so it ends up holding the
**last** match rather than the first. Same output today, different answer
the moment there are two matches.

That is the shape to recognise. "Find the first X" is a loop with a
`break`; leave the `break` out and you have quietly written "find the last
X" instead.

In [ ]:
looked_at = 0
for name in names:
    looked_at = looked_at + 1
    if name.startswith("c"):
        print("found", name, "after", looked_at, "names")
        break

# The same search with no break -- it has the answer and keeps going anyway.
looked_at_all = 0
for name in names:
    looked_at_all = looked_at_all + 1
    if name.startswith("c"):
        found = name

print("without break, looked at", looked_at_all)

### Question 2

`for` … `else`. -> `no match` from the first loop, `found cai` from the second.

The `else` on a loop runs **if the loop was not broken out of**. Read it as
`nobreak:` and it makes sense; read it as the `else` of an `if` and it
never will.

That is exactly the search-failed branch, and it is why the clause exists.
The alternative is a `found = False` flag set before the loop and checked
after — three extra lines that do the same job. Use whichever your team
reads more easily; just be able to recognise both.

Note where the `else` sits: **lined up with `for`**. Indent it four more
spaces and it becomes the `else` of the `if` inside the loop, which is
legal Python, runs on every non-matching name, and means something entirely
different.

In [ ]:
for name in names:
    if name.startswith("z"):
        print("found", name)
        break
else:
    print("no match")          # lines up with `for`, not with `if`

for name in names:
    if name.startswith("c"):
        print("found", name)
        break
else:
    print("no match")

### Question 3

`while` … `else`. -> `3 2 1 countdown complete`; then `3 2 1` and **no** `countdown complete`, ending with `n = 1`.

Same rule on `while`: the `else` runs when the **condition** ends the loop,
and is skipped when a `break` does.

The second version breaks at `n == 1` — before the decrement — so `n` never
reaches 0 and the loop never ends the normal way. The `else` is skipped and
`n` is left at 1.

That left-over value is the useful part. After a broken loop the loop
variable tells you *where* it stopped, which is often the thing you were
looking for. After a completed loop it tells you where the collection ran
out, which usually is not.

In [ ]:
n = 3
while n > 0:
    print(n)
    n = n - 1
else:
    print("countdown complete")

print("---")

n = 3
while n > 0:
    print(n)
    if n == 1:
        break
    n = n - 1
else:
    print("countdown complete")

print("ended with n =", n)

PART B — continue: skip to the next pass

### Question 4

Skipping negatives with `continue`. -> `32.5 from 3 rows; 2 skipped`.

10.0 + 0.0 + 22.5 = 32.5. The two negative rows never reach the total,
because `continue` abandons the rest of the body and starts the next pass.

Put this next to worksheet 04 Q5, which met a value that did not fit and
**stopped**. Here we **skip** and carry on. Neither is more correct — they
are answers to different questions, and the code does not say which
question was asked. *Process until the first failure* and *process
everything valid* produce different numbers from the same rows.

Which is why `skipped` is printed. A total on its own is not a result; a
total plus how many rows did not make it into it is. **If a loop drops
rows, count them and say so** — otherwise 32.5 looks like the total of
everything.

In [ ]:
total = 0.0
kept = 0
skipped = 0

for row in rows:
    if row["amount"] < 0:
        skipped = skipped + 1
        continue          # straight to the next row -- nothing below runs
    total = total + row["amount"]
    kept = kept + 1

print(total, "from", kept, "rows;", skipped, "skipped")

### Question 5

`continue` in a `while`. -> `0`, `1`, then `passes: 20 -- i ended on 2`. **The guard stopped it, not the condition.**

`continue` jumps back to the condition, skipping everything below it — and
everything below it included `i = i + 1`. So `i` stuck at 2, `i < 5` stayed
true forever, and the loop ran until the guard cut it off at 20.

This is the single most common `while` bug there is, and it does not exist
in a `for` loop: `for` advances the iterator itself, so `continue` cannot
stop it moving on.

Two fixes. Increment at the **top** of the body, before any `continue` can
skip it — or just use a `for` loop, which is the better answer nine times
out of ten.

In [ ]:
i = 0
passes = 0

while i < 5 and passes < 20:
    passes = passes + 1
    if i == 2:
        continue          # jumps back to the condition -- SKIPPING the line below
    print(i)
    i = i + 1

print("passes:", passes, "-- i ended on", i)

PART C — pass: do nothing, on purpose

### Question 6

`pass` in a loop. -> `saw 1`, `saw 2`, `saw 3`, `saw 4`, then `4 lines`.

All four printed. `pass` did nothing whatsoever — it is not "skip this
item", it is a statement that exists only so the `if` block is not empty.

Python has no `{}` braces, so a block cannot be empty; `if n % 2 == 0:`
with nothing under it is a `SyntaxError`. `pass` is the placeholder that
makes the block legal while you decide what goes in it.

That is its whole job: a shape you have not filled in yet, or a branch you
deliberately want to do nothing.

In [ ]:
seen = 0
for n in [1, 2, 3, 4]:
    if n % 2 == 0:
        pass
    print("saw", n)
    seen = seen + 1

print(seen, "lines")

### Question 7

`continue` in the same loop. -> `saw 1`, `saw 3`, then `2 lines`. **Half the output of Q6, from a one-word change.**

| | what it does | what runs next |
|---|---|---|
| `pass` | nothing at all | the next line in the body |
| `continue` | abandons this pass | the next item in the loop |
| `break` | abandons the loop | the line after the loop |

`pass` is a placeholder. `continue` is control flow. They read the same in
English — "skip this one" — and swapping them here changes four lines of
output into two, silently.

At the very **end** of a loop body they do happen to behave identically,
which is where the confusion comes from. Anywhere else they do not.

In [ ]:
seen = 0
for n in [1, 2, 3, 4]:
    if n % 2 == 0:
        continue
    print("saw", n)
    seen = seen + 1

print(seen, "lines")

# pass does NOTHING and execution carries straight on to the next line, so
# the rest of the loop body still runs. continue ABANDONS the rest of the
# body and jumps to the next pass. pass is a placeholder; continue is control
# flow.

PART D — Nested loops: break only gets you out of one

### Question 8

`break` inside a nested loop. -> `found 5`, then `outer ran 3 times; inner ran 8 times`.

`break` leaves **one** loop — the innermost one containing it. The inner
loop stopped on row 2; the outer loop then carried on to row 3 as if
nothing had happened.

The counts show it. 3 + 2 + 3 = 8 inner passes: three on row 1, two on row
2 (stopping at the 5), three more on row 3 — after the answer was already
found.

The print says `found 5` exactly once, so the output looks completely
correct. Only the counters reveal that a third of the work was wasted.
On a 3×3 grid that is invisible; inside a nested scan over two tables it is
the whole runtime.

In [ ]:
outer = 0
inner = 0

for row in grid:
    outer = outer + 1
    for value in row:
        inner = inner + 1
        if value == 5:
            print("found 5")
            break

print("outer ran", outer, "times; inner ran", inner, "times")

### Question 9

Breaking out of both. -> `found: True`, then `outer ran 2 times; inner ran 5 times`.

5 inner passes against Q8's 8, and 2 outer against 3. The flag carries the
information *out* of the inner loop so the outer one can act on it — one
variable and one extra `if`.

Python has no `break 2` and no labelled loops. The flag is the standard
answer, and there are two others worth knowing: put the search in a
function and `return` (which leaves everything at once — that is tomorrow's
lecture), or flatten the grid first, as in worksheet 03 Q9, so there is
only one loop to break.

Note `found` does double duty: it exits the outer loop **and** tells the
code afterwards whether the search succeeded. Q8 could not answer that
second question at all.

In [ ]:
outer = 0
inner = 0
found = False

for row in grid:
    outer = outer + 1
    for value in row:
        inner = inner + 1
        if value == 5:
            found = True
            break
    if found:
        break

print("found:", found)
print("outer ran", outer, "times; inner ran", inner, "times")

PART E — All three at once

### Question 10

All three together. -> `sentinel reached`, then `16.5 from 3 values; 1 skipped; 2 never reached`. **The `else` never ran.**

4.0 + 9.5 + 3.0 = 16.5. The `-2.0` was skipped by `continue`, the `"END"`
stopped the loop, and `100.0` and `200.0` were never looked at — 300.0 of
feed sitting past the sentinel, absent from the total, and no error.

The `else` was skipped because the loop broke. Its message would have been
`reached the end of the feed`, and it would have been a lie here; that is
precisely why the clause is useful. **The `else` is how a loop tells you it
finished rather than gave up.**

Which is why all four numbers are printed. `16.5` on its own is
indefensible — it is the total of three of seven entries. `16.5, 3 used, 1
skipped, 2 never reached` is a result you can actually act on, and the
action is probably "go and find out why there is data after the sentinel".

In [ ]:
total = 0.0
used = 0
skipped = 0
seen = 0

for entry in feed:
    seen = seen + 1
    if entry == "END":
        print("sentinel reached")
        break
    if entry < 0:
        skipped = skipped + 1
        continue
    total = total + entry
    used = used + 1
else:
    print("reached the end of the feed")

print(total, "from", used, "values;", skipped, "skipped;",
      len(feed) - seen, "never reached")

### Question 11

A loop variable from a loop that never ran. -> `after a loop that ran, entry_1 is 20`, then `NameError: name 'entry_2' is not defined`.

The first loop ran twice, so `entry_1` exists afterwards and holds the last
value — worksheet 03 Q10's leak, being useful for once.

The second loop ran **zero** times, so `entry_2` was never assigned. It is
not empty and not `None`; it does not exist.

This is the third appearance of the same silence: worksheet 02's
`range(5, 0)`, worksheet 04's condition that started false, and now an
empty list. Each time the loop "worked" and did nothing, and each time the
real damage is downstream — anything the loop was supposed to set is
missing.

Set your variables **before** the loop, not inside it. `total = 0` and
`found = False` on the line above `for` cost nothing and turn a `NameError`
into a correct answer for the empty case.

In [ ]:
for entry_1 in [10, 20]:
    pass
print("after a loop that ran, entry_1 is", entry_1)

for entry_2 in []:
    pass

# This is SUPPOSED to raise: NameError. The loop body never ran, so the loop
# variable was never assigned and does not exist.
#
# Worksheet 04 Q9 was the same silence in a while loop: a condition that
# starts false runs the body zero times. Here it is an empty collection.
# Either way the loop "worked" and did nothing -- and any variable you were
# relying on the loop to set is simply not there.
print(entry_2)